# Embeddings

Loads the same preprocessed GoEmotions dataset as Project 1 (via `projects/src/data.py`).

In [3]:
# Setup: make the course code (part1) and our shared code (projects/src) importable.
# Run from projects/project2 (VS Code / JupyterLab do this by default).
import sys
from pathlib import Path

PROJECT2_DIR = Path.cwd().resolve()
REPO_ROOT = PROJECT2_DIR.parents[1]
PROJECTS_DIR = REPO_ROOT / "projects"
for p in (REPO_ROOT, PROJECTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import pandas as pd

from src import config, data

## Load preprocessed dataset

In [5]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)

df = pd.DataFrame(pairs, columns=["text", "label"])
df["label_name"] = df["label"].map(outcome_space.name)
print(len(df), outcome_space)
df.head()

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}
45446 0: admiration, 1: amusement, 2: anger, 3: annoyance, 4: approval, 5: caring, 6: confusion, 7: curiosity, 8: desire, 9: disappointment, 10: disapproval, 11: disgust, 12: embarrassment, 13: excitement, 14: fear, 15: gratitude, 16: grief, 17: joy, 18: love, 19: nervousness, 20: optimism, 21: pride, 22: realization, 23: relief, 24: remorse, 25: sadness, 26: surprise, 27: neutral


,text,label,label_name
0,Fast as [NAME] will carry me. Seriously uptown...,4,approval
1,You blew it. They played you like a fiddle.,27,neutral
2,TL;DR No more Superbowls for [NAME]. Get ready...,9,disappointment
3,So much time saved. Not.,7,curiosity
4,Emotes have a ridiculous amount of effort put ...,27,neutral


## Pretrained embeddings: GloVe Twitter (27B tokens, 1.2M vocab)

The downloaded zip holds four files (25d / 50d / 100d / 200d), but `part2.clslib.PretrainedEmbeddings` only accepts a zip with exactly one file. So we extract the size we want once into `data/cache/` (git-ignored) and load the `.txt` from there.

In [6]:
import zipfile

import part2.clslib

GLOVE_ZIP = Path.home() / "Downloads" / "archive (2).zip"   # GloVe Twitter zip with all 4 sizes
CACHE_DIR = PROJECT2_DIR / "data" / "cache"                  # git-ignored


def download_pretrained(dims=200):
    """
    Like the tutorial's download_pretrained, but for GloVe Twitter.
    Extracts glove.twitter.27B.{dims}d.txt from GLOVE_ZIP (skipped if already there).
    Returns (path, dims, skip_first_line) for PretrainedEmbeddings.
    """
    if dims not in (25, 50, 100, 200):
        raise ValueError(f"GloVe Twitter has 25/50/100/200 dims, not {dims}")
    name = f"glove.twitter.27B.{dims}d.txt"
    destination = CACHE_DIR / name
    if not destination.is_file():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(GLOVE_ZIP) as archive:
            archive.extract(name, CACHE_DIR)
    return str(destination), dims, False   # GloVe files have no header line

In [ ]:
%%time
embtable_version, embtable_dims, embtable_metadata = download_pretrained(200)
embtable_version, embtable_dims, embtable_metadata

In [ ]:
%%time
# a few minutes for the 200d file (1.2M words)
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

In [ ]:
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}")
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

## MLP classifier (embed-and-pool → MLP → log-softmax)

### Tokenizer

In [7]:
from collections import Counter
from part2 import clslib
import nltk

nltk.download("punkt_tab", quiet=True)

tokenizer = clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
)

# Check how the tokenizer handles Reddit comments
for text, label in pairs[:5]:
    print("Original:", text)
    print("Tokens:  ", tokenizer.tokens(text))
    print()

Original: Fast as [NAME] will carry me. Seriously uptown to downtown via Claiborne or broad isn’t that bad
Tokens:   ['fast', 'as', '[', 'name', ']', 'will', 'carry', 'me', '.', 'seriously', 'uptown', 'to', 'downtown', 'via', 'claiborne', 'or', 'broad', 'isn', '’', 't', 'that', 'bad']

Original: You blew it. They played you like a fiddle.
Tokens:   ['you', 'blew', 'it', '.', 'they', 'played', 'you', 'like', 'a', 'fiddle', '.']

Original: TL;DR No more Superbowls for [NAME]. Get ready for another winning season that ends in disappointment.
Tokens:   ['tl', ';', 'dr', 'no', 'more', 'superbowls', 'for', '[', 'name', ']', '.', 'get', 'ready', 'for', 'another', 'winning', 'season', 'that', 'ends', 'in', 'disappointment', '.']

Original: So much time saved. Not.
Tokens:   ['so', 'much', 'time', 'saved', '.', 'not', '.']

Original: Emotes have a ridiculous amount of effort put into them, which only makes the fact that nobody uses them sadder
Tokens:   ['emotes', 'have', 'a', 'ridiculous', 'am

In [8]:
token_counts = Counter()

for text, label in pairs:
    token_counts.update(tokenizer.tokens(text))

vocab = set(token_counts)

print("Unique tokens:", len(vocab))
print("Total token occurrences:", sum(token_counts.values()))
print("\nMost frequent tokens:")
print(token_counts.most_common(20))

Unique tokens: 29784
Total token occurrences: 697506

Most frequent tokens:
[('.', 37371), ('i', 20475), ('the', 18563), (',', 14413), ('to', 12961), ('a', 12831), ('you', 11402), ('it', 10959), ('that', 9935), ('and', 9227), ('is', 9022), ('!', 8806), ('name', 8489), ('[', 8426), (']', 8424), ('of', 7398), ('’', 6819), ('this', 6591), ('in', 6088), ('for', 5789)]


### Encoder (average of word embeddings)

### Model (MLP on top of the encoder)

### Cross-validation splits

### Training

### Evaluation